In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
df = spark.read.format('csv').option('inferSchema',True).option('header',True).load("/Volumes/workspace/new/volume1/BigMart Sales.csv")

In [0]:
df.display()

In [0]:
df.printSchema()

In [0]:
my_ddl_schema = '''
                    Item_Identifier STRING,
                    Item_Weight STRING,
                    Item_Fat_Content STRING, 
                    Item_Visibility DOUBLE,
                    Item_Type STRING,
                    Item_MRP DOUBLE,
                    Outlet_Identifier STRING,
                    Outlet_Establishment_Year INT,
                    Outlet_Size STRING,
                    Outlet_Location_Type STRING, 
                    Outlet_Type STRING,
                    Item_Outlet_Sales DOUBLE 
'''

In [0]:
df = spark.read.format('csv').schema(my_ddl_schema).option('header',True).load("/Volumes/workspace/new/volume1/BigMart Sales.csv")
df.printSchema()
df.display()


#### StructType()

In [0]:
schema = StructType([
    StructField("Item_Identifier", StringType(), True),
    StructField("Item_Weight", StringType(), True),
    StructField("Item_Fat_Content", StringType(), True),
    StructField("Item_Visibility", DoubleType(), True),
    StructField("Item_Type", StringType(), True),
    StructField("Item_MRP", DoubleType(), True),
    StructField("Outlet_Identifier", StringType(), True),
    StructField("Outlet_Establishment_Year", IntegerType(), True),
    StructField("Outlet_Size", StringType(), True),
    StructField("Outlet_Location_Type", StringType(), True),
    StructField("Outlet_Type", StringType(), True),
    StructField("Item_Outlet_Sales", DoubleType(), True)
])

In [0]:
df_new = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load("/Volumes/workspace/new/volume1/BigMart Sales.csv")

In [0]:
df_new.printSchema()
df_new.display

## Transformations

### Select
##### Used to pick columns and optionally transform them.select() returns only the columns you specify.Create new dataframe with few columns

In [0]:
df.select(col("Item_MRP"),col("Item_Type"),col("Item_Weight")).show()

### Alias

In [0]:
df_id = df.select(col("Item_Identifier").alias('Item_ID'))
df_id.display()

In [0]:
df.display()

### FILTER

#### Scenario-1

In [0]:
df.filter(col("Item_Fat_Content")=='Regular').display()

#### Scenario-2

In [0]:
df.filter((col("Item_Weight").cast("float")<10) & (col("Item_Type") == 'Soft Drinks')).display()

### withColumnRenamed

In [0]:
df.withColumnRenamed('Item_Weight','Item_Wt').display()

### withColumn()

 withColumn() is used to add a new column or modify an existing column in a DataFrame.

 ```
 df.withColumn("new_column_name", expression)
 ```

In [0]:
df_new = df.withColumn(
    "MRP_with_tax",
    col("Item_MRP") * 1.18
)

df_new.display()

In [0]:
df = df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Regular","Reg"))\
    .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Low Fat","Lf"))

df.display()

### Type Casting

In [0]:
df = df.withColumn('Item_Weight',col("Item_Weight").cast(StringType()))

In [0]:
df.printSchema()

### sort
sort() is used to order rows in a DataFrame based on one or more columns.
It is similar to SQL ORDER BY.
```
df.sort("column_name")

```

In [0]:
display(df.sort("Item_MRP"))

In [0]:
display(df.sort(desc("Item_MRP")))

In [0]:
display(
    df.sort(['Item_Weight','Item_Visibility'], ascending=[False, False])
)

In [0]:
df.sort(['Item_weight','Item_Visibility'], ascending=[0,1]).display() 

# Item_weight → sorted descending (0 = False)

# Item_Visibility → sorted ascending (1 = True)

### Limit

In [0]:
df.limit(10).display()

### **DROP**

drop() is used to remove one or more columns from a DataFrame.
It returns a new DataFrame without those columns (original DataFrame remains unchanged).

In [0]:
df2 = df.drop("Item_Weight", "Outlet_Size")

display(df2)

### Drop duplicates

In [0]:
df3 = df.groupBy("Item_Identifier").count()



In [0]:
display(df3)

### dropDuplicates
dropDuplicates() removes rows that are completely identical across all columns

In [0]:
df.dropDuplicates().display()

In [0]:
#This keeps unique combinations.
df.dropDuplicates(["Item_Identifier","Outlet_Identifier"]).display()

### UNION and UNION BY NAME
union() combines two DataFrames based on column position, not column names.
unionByName() combines DataFrames based on column names.

In [0]:
data1 = [('1','ABC'),
          ('2','DEF'),
          ('3','GHI'),
          ('4','JKL')]
schema1 = 'id STRING, name STRING'

df1 = spark.createDataFrame(data1,schema1)

In [0]:
data2 = [('5','ABC'),
          ('6','DEF'),
          ('7','GHI'),
          ('8','JKL')]
schema2 = 'id STRING, name STRING'

df2 = spark.createDataFrame(data2,schema2)

In [0]:
df1.display()
df2.display()

### Union

In [0]:
df1.union(df2).display()

In [0]:
data3 = [('kad','1',),
        ('sid','2',)]
schema3 = 'name STRING, id STRING' 

df3 = spark.createDataFrame(data3,schema3)

df3.display()

In [0]:
df1.union(df3).display()

### Union by Name

In [0]:
df1.unionByName(df3).display()

### String Functions


String functions in PySpark are used to manipulate and process text data in DataFrame columns.

#### Import Functions

```python
from pyspark.sql.functions import *
```
#### 1. upper()

Converts text to uppercase.

```python
df.select(upper("Item_Type")).show()
```

Example

```
dairy → DAIRY
```

#### 2. lower()

Converts text to lowercase.

```python
df.select(lower("Item_Type")).show()
```

Example

```
DAIRY → dairy
```
#### 3. length()

Returns the number of characters in a string.

```python
df.select(length("Item_Identifier")).show()
```

Example

```
FDA15 → 5
```

#### 4. concat()

Combines multiple columns into one string.

```python
df.select(concat("Item_Identifier", "Item_Type")).show()
```
#### 5. concat_ws()

Concatenates columns with a separator.

```python
df.select(concat_ws("-", "Item_Identifier", "Item_Type")).show()
```

Example

```
FDA15-Dairy
```
#### 6. substring()

Extracts part of a string.

```python
df.select(substring("Item_Identifier", 1, 3)).show()
```

Example

```
FDA15 → FDA
```
#### 7. trim()

Removes spaces from both sides of a string.

```python
df.select(trim("Item_Type")).show()
```

Related functions

```
ltrim() → remove left spaces
rtrim() → remove right spaces
```
#### 8. regexp_replace()

Replaces text using a regular expression.

```python
df.select(regexp_replace("Item_Type", "Dairy", "Milk")).show()
```
#### 9. split()

Splits a string into an array



In [0]:
df.select(upper("Item_Type")).show()

In [0]:
df.select(length("Item_Identifier")).show()
df.select(concat("Item_Identifier", "Item_Type")).show()
df.select(concat_ws("-", "Item_Identifier", "Item_Type")).show()
df.select(substring("Item_Identifier", 1, 3)).show()
df.select(trim("Item_Type")).show()
df.select(regexp_replace("Item_Type", "Dairy", "Milk")).show()
df.select(split("Item_Identifier", "A")).show()
df.filter(col("Item_Type").contains("Dairy")).show()
df.filter(col("Item_Identifier").startswith("FD")).show()
df.filter(col("Item_Identifier").endswith("5")).show()
df.select(initcap("Item_Type")).show()

### Date Functions



Date functions in PySpark are used to manipulate and analyze date and timestamp columns.


#### 1. current_date()

Returns the current system date.

```python
df.select(current_date()).show()
```

Example

```
2026-03-11
```

---

#### 2. current_timestamp()

Returns the current system timestamp.

```python
df.select(current_timestamp()).show()
```

Example

```
2026-03-11 15:45:21
```

---

#### 3. to_date()

Converts a string column into a date.

```python
df.select(to_date("date_column")).show()
```

Example

```
"2026-03-11" → 2026-03-11
```

---

#### 4. to_timestamp()

Converts a string column to timestamp.

```python
df.select(to_timestamp("date_column")).show()
```

---

#### 5. date_format()

Formats a date column.

```python
df.select(date_format("date_column", "yyyy-MM-dd")).show()
```

Example

```
2026-03-11 → 11-03-2026
```

---

#### 6. year()

Extracts the year from a date.

```python
df.select(year("date_column")).show()
```

Example

```
2026-03-11 → 2026
```

---

#### 7. month()

Extracts the month from a date.

```python
df.select(month("date_column")).show()
```

Example

```
2026-03-11 → 3
```

---

#### 8. dayofmonth()

Extracts the day of the month.

```python
df.select(dayofmonth("date_column")).show()
```

Example

```
2026-03-11 → 11
```

---

#### 9. dayofweek()

Returns the day of the week.

```python
df.select(dayofweek("date_column")).show()
```

Example

```
Sunday = 1
Monday = 2
```

---

#### 10. datediff()

Returns the difference between two dates.

```python
df.select(datediff("date1", "date2")).show()
```

Example

```
2026-03-11 - 2026-03-01 = 10
```

---

#### 11. date_add()

Adds days to a date.

```python
df.select(date_add("date_column", 5)).show()
```

Example

```
2026-03-11 → 2026-03-16
```

---

#### 12. date_sub()

Subtracts days from a date.

```python
df.select(date_sub("date_column", 5)).show()
```

Example

```
2026-03-11 → 2026-03-06
```

---

#### 13. add_months()

Adds months to a date.

```python
df.select(add_months("date_column", 2)).show()
```

Example

```
2026-03-11 → 2026-05-11
```

---

#### 14. last_day()

Returns the last day of the month.

```python
df.select(last_day("date_column")).show()
```

Example

```
2026-03-11 → 2026-03-31
```

---

#### Most Commonly Used Date Functions in ETL Pipelines

```
current_date()
current_timestamp()
to_date()
year()
month()
datediff()
date_add()
date_sub()
add_months()
last_day()
```



In [0]:
 df.select(current_date()).show()
 df.select(current_timestamp()).display()
 


### Null Values